# 06 -- Figure Export for Report

Exports all key figures as high-resolution PNG files to `report/figures/`
for inclusion in the written report.

In [1]:
# Setup

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "iframe"

import warnings
warnings.filterwarnings("ignore")

from src.data_utils import load_processed
from src.nig import NIGParams, nig_pdf, nig_cdf
from src.assessment import christoffersen_test, anderson_darling
from scipy import stats

# Output directory
FIGURES = Path("../report/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name, width=1000, height=500):
    path = FIGURES / f"{name}.png"
    fig.write_image(str(path), scale=2, width=width, height=height)
    print(f"Saved → {path}")

print("Setup OK")

Setup OK


In [2]:
# Load all data

innovations    = load_processed("innovations.parquet")
sigma_forecasts = load_processed("sigma_forecasts.parquet")
nig_params_df  = load_processed("nig_params.parquet")
var99          = load_processed("var99.parquet")
var95          = load_processed("var95.parquet")
master_df      = load_processed("master_assessment.parquet")
exceedance_df  = load_processed("exceedance_results.parquet")

TICKERS = list(innovations.columns)
colors  = dict(zip(TICKERS, px.colors.qualitative.Plotly))

nig_params = {}
for ticker in TICKERS:
    row = nig_params_df.loc[ticker]
    nig_params[ticker] = NIGParams(
        alpha=row["alpha"], beta=row["beta"],
        mu=row["mu"],       delta=row["delta"],
    )

all_results = {}
for ticker in TICKERS:
    safe = ticker.replace("^","").replace("=","_")
    all_results[ticker] = load_processed(f"rolling_{safe}.parquet")

print("Data loaded")

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\innovations.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\sigma_forecasts.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\nig_params.parquet  shape=(5, 4)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\var99.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch

## Figure 1 - Return Distributions vs Gaussian

In [3]:
fig = make_subplots(rows=1, cols=len(TICKERS),
                    subplot_titles=TICKERS,
                    horizontal_spacing=0.04)

x_grid   = np.linspace(-7, 7, 400)
gauss_ref = stats.norm.pdf(x_grid)

for i, ticker in enumerate(TICKERS):
    innov = innovations[ticker].dropna().values
    fig.add_trace(go.Histogram(
        x=innov, nbinsx=60, histnorm="probability density",
        marker_color=colors[ticker], opacity=0.5,
        showlegend=False,
    ), row=1, col=i+1)
    fig.add_trace(go.Scatter(
        x=x_grid, y=gauss_ref, mode="lines",
        line=dict(color="black", width=1.5, dash="dash"),
        name="Gaussian" if i==0 else None,
        showlegend=(i==0),
    ), row=1, col=i+1)
    fig.update_xaxes(range=[-6,6], row=1, col=i+1)

fig.update_layout(
    template="plotly_white", height=340,
    margin=dict(t=40, b=40, l=40, r=20),
    font=dict(size=11),
)
save_fig(fig, "fig1_return_distributions", width=1200, height=340)
fig.show()

Saved → ..\report\figures\fig1_return_distributions.png


## Figure 2: NIG vs Student T vs Gaussian fits:

In [4]:
x_grid    = np.linspace(-7, 7, 500)
gauss_ref = stats.norm.pdf(x_grid)

fig = make_subplots(rows=1, cols=len(TICKERS),
                    subplot_titles=TICKERS,
                    horizontal_spacing=0.04)

for i, ticker in enumerate(TICKERS):
    innov  = innovations[ticker].dropna().values
    p      = nig_params[ticker]
    df_t, loc_t, scale_t = stats.t.fit(innov)

    nig_v = nig_pdf(x_grid, p)
    t_v   = stats.t.pdf(x_grid, df=df_t, loc=loc_t, scale=scale_t)

    fig.add_trace(go.Histogram(
        x=innov, nbinsx=60, histnorm="probability density",
        marker_color=colors[ticker], opacity=0.35,
        showlegend=False,
    ), row=1, col=i+1)
    fig.add_trace(go.Scatter(
        x=x_grid, y=nig_v, mode="lines",
        line=dict(color="#e63946", width=2),
        name="NIG" if i==0 else None, showlegend=(i==0),
    ), row=1, col=i+1)
    fig.add_trace(go.Scatter(
        x=x_grid, y=t_v, mode="lines",
        line=dict(color="#f4a261", width=2, dash="dot"),
        name="Student T" if i==0 else None, showlegend=(i==0),
    ), row=1, col=i+1)
    fig.add_trace(go.Scatter(
        x=x_grid, y=gauss_ref, mode="lines",
        line=dict(color="#457b9d", width=1.5, dash="dash"),
        name="Gaussian" if i==0 else None, showlegend=(i==0),
    ), row=1, col=i+1)
    fig.update_xaxes(range=[-6,6], row=1, col=i+1)

fig.update_layout(
    template="plotly_white", height=360,
    legend=dict(orientation="h", y=-0.15),
    margin=dict(t=40, b=60, l=40, r=20),
    font=dict(size=11),
)
save_fig(fig, "fig2_distribution_fits", width=1200, height=360)
fig.show()

Saved → ..\report\figures\fig2_distribution_fits.png


## Figure 3: VaR vs actual returns (^GSPC only, cleaner for report):

In [6]:
# --- Figure 3a: Combined panel (main body) ---
fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=TICKERS,
    vertical_spacing=0.06,
)

for i, ticker in enumerate(TICKERS):
    actual  = all_results[ticker]["return"]
    var99_s = var99[ticker]
    var95_s = var95[ticker]
    exceed  = actual < var99_s

    fig.add_trace(go.Scatter(
        x=actual.index, y=actual.values,
        mode="lines", name=f"{ticker}",
        line=dict(color=colors[ticker], width=0.7),
        showlegend=False,
    ), row=i+1, col=1)

    fig.add_trace(go.Scatter(
        x=var99_s.index, y=var99_s.values,
        mode="lines",
        name="VaR 99%" if i==0 else None,
        showlegend=(i==0),
        line=dict(color="#e63946", width=1.5),
    ), row=i+1, col=1)

    fig.add_trace(go.Scatter(
        x=var95_s.index, y=var95_s.values,
        mode="lines",
        name="VaR 95%" if i==0 else None,
        showlegend=(i==0),
        line=dict(color="#c47000", width=1.2, dash="dot"),
    ), row=i+1, col=1)

    fig.add_trace(go.Scatter(
        x=actual.index[exceed], y=actual.values[exceed],
        mode="markers",
        name="Exceedance" if i==0 else None,
        showlegend=(i==0),
        marker=dict(color="#040720", size=5, symbol="x"),
    ), row=i+1, col=1)

    fig.update_yaxes(title_text="Log ret.", row=i+1, col=1,
                     title_font=dict(size=9))

fig.update_layout(
    template="plotly_white",
    height=1100,
    legend=dict(orientation="h", y=-0.03),
    margin=dict(t=40, b=60, l=55, r=20),
    font=dict(size=10),
)
save_fig(fig, "fig3_var_all_assets", width=1000, height=1100)
fig.show()
print("Combined panel saved")

# --- Figures 3b–3f: Individual assets (appendix) ---
for ticker in TICKERS:
    actual  = all_results[ticker]["return"]
    var99_s = var99[ticker]
    var95_s = var95[ticker]
    exceed  = actual < var99_s

    fig_ind = go.Figure()
    fig_ind.add_trace(go.Scatter(
        x=actual.index, y=actual.values,
        mode="lines", name="Log return",
        line=dict(color=colors[ticker], width=0.7),
    ))
    fig_ind.add_trace(go.Scatter(
        x=var99_s.index, y=var99_s.values,
        mode="lines", name="VaR 99%",
        line=dict(color="#e63946", width=1.5),
    ))
    fig_ind.add_trace(go.Scatter(
        x=var95_s.index, y=var95_s.values,
        mode="lines", name="VaR 95%",
        line=dict(color="#c47000", width=1.2, dash="dot"),
    ))
    fig_ind.add_trace(go.Scatter(
        x=actual.index[exceed], y=actual.values[exceed],
        mode="markers", name="Exceedance",
        marker=dict(color="#040720", size=6, symbol="x"),
    ))
    fig_ind.update_layout(
        title=f"VaR vs Actual Returns — {ticker}",
        template="plotly_white",
        xaxis_title="Date",
        yaxis_title="Log return",
        legend=dict(orientation="h", y=-0.18),
        height=380,
        margin=dict(t=40, b=70, l=50, r=20),
        font=dict(size=11),
    )
    safe = ticker.replace("^","").replace("=","_")
    save_fig(fig_ind, f"fig3_var_{safe}", width=1000, height=380)
    print(f"Individual figure saved --> {ticker}")

Saved → ..\report\figures\fig3_var_all_assets.png


Combined panel saved
Saved → ..\report\figures\fig3_var_AAPL.png
Individual figure saved --> AAPL
Saved → ..\report\figures\fig3_var_EURUSD_X.png
Individual figure saved --> EURUSD=X
Saved → ..\report\figures\fig3_var_GLD.png
Individual figure saved --> GLD
Saved → ..\report\figures\fig3_var_TIP.png
Individual figure saved --> TIP
Saved → ..\report\figures\fig3_var_GSPC.png
Individual figure saved --> ^GSPC


## Figure 4: VaR backtesting table (both levels combined):

In [7]:
color_map = {"GREEN": "#2dc653", "RED": "#e63946", "BLUE": "#457b9d"}
font_map  = {"GREEN": "#144a23", "RED": "#5c0a10", "BLUE": "#0d2b40"}

exc = exceedance_df.copy()

fig = go.Figure(data=[go.Table(
    columnwidth=[90, 60, 80, 80, 70, 80],
    header=dict(
        values=["Ticker", "VaR level", "Expected",
                "Exceedances", "p-value", "Result"],
        fill_color="#2c3e50",
        font=dict(color="white", size=12),
        align="center", height=32,
    ),
    cells=dict(
        values=[
            exc["Ticker"].tolist(),
            exc["VaR level"].tolist(),
            exc["Expected"].tolist(),
            exc["Exceedances"].tolist(),
            exc["p-value"].tolist(),
            exc["Result"].tolist(),
        ],
        fill_color=[
            ["#f8f9fa"] * len(exc),
            ["#f8f9fa"] * len(exc),
            ["#f8f9fa"] * len(exc),
            ["#f8f9fa"] * len(exc),
            ["#f8f9fa"] * len(exc),
            [color_map[r] for r in exc["Result"]],
        ],
        font=dict(
            color=[
                ["#2c3e50"] * len(exc),
                ["#2c3e50"] * len(exc),
                ["#2c3e50"] * len(exc),
                ["#2c3e50"] * len(exc),
                ["#2c3e50"] * len(exc),
                [font_map[r] for r in exc["Result"]],
            ],
            size=12,
        ),
        align="center", height=28,
    ),
)])
fig.update_layout(
    template="plotly_white", height=400,
    margin=dict(t=20, b=10, l=10, r=10),
)
save_fig(fig, "fig4_var_backtest_table", width=900, height=400)
fig.show()

Saved → ..\report\figures\fig4_var_backtest_table.png


## Figure 5: Master assessment table:

In [8]:
m = master_df.copy()
m.index.name = "Ticker"
m = m.reset_index()

color_map = {"GREEN": "#2dc653", "RED": "#e63946", "BLUE": "#457b9d"}

fig = go.Figure(data=[go.Table(
    columnwidth=[80, 65, 70, 75, 65, 70, 75, 65, 65, 75, 55],
    header=dict(
        values=["Ticker","VaR99<br>Exceed","VaR99<br>p-val",
                "VaR99<br>Result","VaR95<br>Exceed","VaR95<br>p-val",
                "VaR95<br>Result","KS<br>(NIG)","AD<br>(NIG)",
                "Christ.<br>p-val","Indep."],
        fill_color="#2c3e50",
        font=dict(color="white", size=10),
        align="center", height=36,
    ),
    cells=dict(
        values=[
            m["Ticker"].tolist(),
            m["VaR99 exceed"].tolist(),
            m["VaR99 p-val"].tolist(),
            m["VaR99 result"].tolist(),
            m["VaR95 exceed"].tolist(),
            m["VaR95 p-val"].tolist(),
            m["VaR95 result"].tolist(),
            m["KS (NIG)"].tolist(),
            m["AD (NIG)"].tolist(),
            m["Christoffersen p"].tolist(),
            m["Indep."].tolist(),
        ],
        fill_color=[
            ["#f8f9fa"]*len(m),
            ["#f8f9fa"]*len(m),
            ["#f8f9fa"]*len(m),
            [color_map.get(r,"#f8f9fa") for r in m["VaR99 result"]],
            ["#f8f9fa"]*len(m),
            ["#f8f9fa"]*len(m),
            [color_map.get(r,"#f8f9fa") for r in m["VaR95 result"]],
            ["#f8f9fa"]*len(m),
            ["#f8f9fa"]*len(m),
            ["#f8f9fa"]*len(m),
            ["#f8f9fa"]*len(m),
        ],
        font=dict(size=10),
        align="center", height=28,
    ),
)])
fig.update_layout(
    template="plotly_white", height=280,
    margin=dict(t=20, b=10, l=10, r=10),
)
save_fig(fig, "fig5_master_table", width=1100, height=280)
fig.show()

Saved → ..\report\figures\fig5_master_table.png


## Figure 6: Sensitivity analysis:

In [9]:
sens_df = load_processed("sensitivity_analysis.parquet")
sens_df = sens_df.reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=sens_df["Window"].astype(str),
    y=sens_df["Exceedances (VaR99)"],
    marker_color=["#2dc653"]*len(sens_df),
    text=sens_df["p-value"].apply(lambda x: f"p={x:.4f}"),
    textposition="outside",
    name="Exceedances",
    width=0.4,
))
fig.add_hline(
    y=10, line_dash="dash",
    line_color="#e63946", line_width=1.5,
    annotation_text="Expected = 10",
    annotation_position="right",
)
fig.update_layout(
    template="plotly_white",
    xaxis_title="Estimation window (days)",
    yaxis_title="VaR 99% exceedances",
    yaxis=dict(range=[0, 16]),
    height=380,
    margin=dict(t=30, b=50, l=50, r=80),
    font=dict(size=12),
    showlegend=False,
)
save_fig(fig, "fig6_sensitivity", width=600, height=380)
fig.show()

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\sensitivity_analysis.parquet  shape=(3, 3)
Saved → ..\report\figures\fig6_sensitivity.png


In [ ]:
## Verify all figures exported:

